In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Set visualization styles
sns.set_theme(style="whitegrid")
print("Environment successfully set up!")

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
n_rows = 5000
# Generate dummy session attributes
session_ids = [f"sess_{i:04d}" for i in range(1, n_rows + 1)]
user_ids = [f"user_{np.random.randint(1000, 2000)}" for _ in range(n_rows)]
device_categories = np.random.choice(['desktop', 'mobile', 'tablet'], size=n_rows, p=[0.5, 0.4, 0.1])
# Funnel flags (session_start is always 1, others follow realistic probabilities)
session_start_flag = np.ones(n_rows, dtype=int)
add_to_cart_flag = np.random.choice([0, 1], size=n_rows, p=[0.7, 0.3]) # 30% add to cart
purchase_flag = np.where(add_to_cart_flag == 1, np.random.choice([0, 1], size=n_rows, p=[0.4, 0.6]), 0)
# 60%conversion from cart
page_views = np.random.poisson(lam=3, size=n_rows) + 1
time_on_site_sec = page_views * np.random.randint(20, 60, size=n_rows)

In [ ]:
# Create DataFrame
df = pd.DataFrame({
'session_id': session_ids,
'user_id': user_ids,
'device_category': device_categories,
'session_start_flag': session_start_flag,
'add_to_cart_flag': add_to_cart_flag,
'purchase_flag': purchase_flag,
'page_views': page_views,
'time_on_site_sec': time_on_site_sec
})
# Introduce artificial missing values in 'time_on_site_sec' for data cleaning practice
df.loc[np.random.choice(n_rows, 50), 'time_on_site_sec'] = np.nan
# Save to CSV and inspect the first 5 rows
df.to_csv('ga4_sample_session_logs.csv', index=False)
print("Dataset 'ga4_sample_session_logs.csv' successfully generated and loaded!")
df.head()

In [ ]:
df.info()

In [ ]:
# Calculate funnel metrics
total_sessions = df['session_id'].nunique()
sessions_with_cart = df[df['add_to_cart_flag'] == 1]['session_id'].nunique()
sessions_with_purchase = df[df['purchase_flag'] == 1]['session_id'].nunique()
# Create a summary DataFrame
funnel_data = {
'Funnel Step': ['1. Session Start', '2. Add to Cart', '3. Purchase'],
'User Sessions': [total_sessions, sessions_with_cart, sessions_with_purchase]
}
funnel_df = pd.DataFrame(funnel_data)
# Calculate conversion rates relative to step 1
funnel_df['Conversion Rate (%)'] = (funnel_df['User Sessions'] / total_sessions) * 100
print(funnel_df)


In [ ]:
print(df.isnull().sum())

In [ ]:
# Impute missing time_on_site_sec with median
median_time = df['time_on_site_sec'].median()
df['time_on_site_sec'] = df['time_on_site_sec'].fillna(median_time)
# Drop rows with missing session IDs if any
df = df.dropna(subset=['session_id'])
# Clean categorical formatting
df['device_category'] = df['device_category'].str.lower().str.strip()
print("Data cleaning completed. Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Select numerical columns relevant to conversion behavior
numeric_cols = ['session_start_flag', 'add_to_cart_flag', 'purchase_flag', 'page_views', 'time_on_site_sec']
corr_matrix = df[numeric_cols].corr()
print(corr_matrix)

In [ ]:
# Plot the correlation heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Digital Behavioral Correlation Heatmap – Identifying Conversion Blockers", fontsize=12)
plt.show()
